In [1]:
import pandas as pd
import numpy as np
import yaml

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.5f}'.format)

with open('../../Settings.yaml', 'r') as file:
    Setting = yaml.safe_load(file)

#Use First Dataset to Create our Needed Data
file_name = "Unadjusted.xlsx"
file_path = f"{Setting['Output_Path_Unajusted']}/{file_name}"
Statistics = pd.read_excel(file_path,sheet_name='Statistics_By_Activity')
Statistics = Statistics[['Year','Industry_Category_Code','Industry_Code',
                         'Industry_Name','Wage_Value','Total_Output_Industrial_Activity_Value','Manufactured_Product_Value',
                         'Employees_Quantity','Total_Value_Added_Industrial_Activity','Saled_Product_Value']]
Statistics = Statistics[Statistics['Industry_Category_Code'] == 2]

#Use Second Dataset to Create our Needed Data
second_file_path = f"{Setting['Output_Path_Unajusted']}/{file_name}"
Input = pd.read_excel(second_file_path,sheet_name='Input_By_Activity')
Input = Input[['Year','Industry_Category_Code','Industry_Code','Input_Value_Without_Energy',
               'Input_Value_Fuel','Input_Value_Electricity','Input_Value_Water']]
Input = Input[Input['Industry_Category_Code'] == 2]

#Use Third Dataset to Create our Needed Data
third_file_name = "Unadjusted.xlsx"
third_file_path = f"{Setting['Output_Path_Unajusted']}/{third_file_name}"
Capital = pd.read_excel(third_file_path,sheet_name = 'Capital_Stock_By_Activity')
Capital = Capital[['Year','Industry_Code','Capital_Stock']]


#Use Fourth Dataset to Create our Needed Data
fourth_file_name = "General.xlsx"
fourth_file_path = f"{Setting['Output_Path_General']}/{fourth_file_name}"
PPI = pd.read_excel(fourth_file_path,sheet_name='PPI')
PPI = PPI[PPI['Industry_Category_Code']==2]
PPI.drop(columns = ['Industry_Category_Code','Industry_Name'],inplace = True)


#Use fifth Dataset to Create our Needed Data
fifth_file_name = "Unadjusted.xlsx"
fifth_file_path = f"{Setting['Output_Path_Unajusted']}/{fifth_file_name}"
Energy_Rate = pd.read_excel(fifth_file_path,sheet_name='Energy_Nominal_Rate')


#Use sixth Dataset to Create our Needed Data
sixth_file_name = "General.xlsx"
sixth_file_path = f"{Setting['Output_Path_General']}/{sixth_file_name}"
Weather = pd.read_excel(sixth_file_path,sheet_name='Weather_Year')

#Use seventh Dataset to Create our Needed Data
seventh_file_name = "Unadjusted.xlsx"
seventh_file_path = f"{Setting['Output_Path_Unajusted']}/{seventh_file_name}"
Energy_Intensity = pd.read_excel(seventh_file_path,sheet_name='Energy_By_Activity')

#Use eigths Dataset to Create our Needed Data
eight_file_name = "Blackout.xlsx"
eight_file_path = f"{Setting['Raw_Path']}/{eight_file_name}"
Blackout = pd.read_excel(eight_file_path)

#Create Our Our Dataset by Merging Datasets
FDataset = pd.merge(
    Statistics,
    Input,
    how ='left',
    on = ['Year','Industry_Code']
    )

FDataset['Industry_Code'] = FDataset['Industry_Code'].astype(int)
PPI['Industry_Code'] = PPI['Industry_Code'].astype(int)

SDataset = pd.merge(
    FDataset,
    PPI,
    how = 'left',
    on = ['Year','Industry_Code']
)

Dataset = pd.merge(
    SDataset,
    Capital,
    how = 'left',
    on = ['Year','Industry_Code']
)

Dataset = pd.merge(
    Dataset,
    Energy_Rate,
    how = 'left',
    on = ['Year','Industry_Code']
)

Dataset = pd.merge(
    Dataset,
    Weather,
    how = 'left',
    on = ['Year']
)

Dataset = pd.merge(
    Dataset,
    Energy_Intensity,
    how = 'left',
    on =  ['Year','Industry_Code']
)

Dataset = pd.merge(
    Dataset,
    Blackout[['Year','Blackout']],
    how = 'left',
    on =  ['Year']
)

#Delete Unwanted Datasets and Columns

del FDataset,SDataset,Input,Statistics,Capital,PPI
Dataset.drop(columns=['Industry_Category_Code_y','Industry_Category_Code_x','Industry_Name_y'],inplace=True)



#Rename And Reorder Columns

Dataset.rename(
    columns = {
        'Industry_Name_x':'Industry_Name',
        'Saled_Product_Value':'Sale',
        'Manufactured_Product_Value' : 'Product',
        'Wage_Value':'Wage',
        'Total_Output_Industrial_Activity_Value':'Output_Value',
        'Input_Value_Without_Energy':'Input_Value',
        'Input_Value_Fuel': 'Fuel',
        'Input_Value_Electricity': 'Electricity',
        'Input_Value_Water':'Water',
        'Capital_Stock':'Capital',
        'Employees_Quantity' : 'Labour_No',
        'Total_Value_Added_Industrial_Activity' : 'Value_Added'
    } ,
    inplace = True)


#Change Nominal Values To Real Values in Rial

cols = ['Output_Value','Wage', 'Input_Value', 'Fuel', 'Sale' , 'Product',
         'Electricity', 'Water', 'Capital','Value_Added',
         'Electricity_unit_price','Water_unit_price']

Dataset[cols] = 10**8 * Dataset[cols].div(Dataset['Price_Index'], axis=0)

Dataset.drop(columns = ['Price_Index','Natural_Gas_unit_price','Gasoil_unit_price',
                        'Gasoline_unit_price','Kerosene_unit_price','LNG_unit_price',
                        'Coal_Charcoal_unit_price','Fueloil_unit_price',
                        'Rainfall', 'SunHours',
                        'Natural_Gas (M3)', 'Fueloil (Liter)', 'Gasoil (Liter)',
                        'Gasoline (Liter)', 'Kerosene (Liter)', 'LNG (KG)',
                        'Charcoal and Coal (KG)', 'Electricity (KWH)', 'Water (M3)',
                        'Electricity (boe)', 'Energy (boe)'],
            inplace = True)



Dataset['Elec_value_intensity'] = Dataset['Electricity'] / (Dataset['Fuel'] + Dataset['Electricity'])
Dataset['Elec_value_share_of_input'] = Dataset['Electricity'] / (Dataset['Input_Value'])


Dataset = Dataset.loc[:, ~Dataset.columns.duplicated()]

Dataset['Log Y/L']= np.log(Dataset['Output_Value']/Dataset['Wage'])
Dataset['Log Y/K']= np.log(Dataset['Output_Value']/Dataset['Capital'])

#Export to Excel
output_file_name = 'Adjusted.xlsx'
sheet_name= 'Dataset_for_Model'
path = f"{Setting['Output_Path_Ajusted']}/{output_file_name}"
with pd.ExcelWriter(path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    Dataset.to_excel(writer,sheet_name=sheet_name ,index=False)